In [ ]:
%%capture
!pip install gradio scikit-learn tensorflow pandas matplotlib seaborn

# 1. Carga de Datos y Entrenamiento de la Red Neuronal

Primero, importaremos las librerías necesarias, cargaremos el dataset Iris, y diseñaremos y entrenaremos la red neuronal utilizando Keras.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Configuración de estilos visuales
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

# 1. Cargar el dataset de Iris
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# Crear DataFrame para el EDA
df = pd.DataFrame(X, columns=feature_names)
df['species'] = [target_names[i] for i in y]

# 2. Preprocesamiento para la Red Neuronal
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

encoder = OneHotEncoder(sparse_output=False)
y_onehot = encoder.fit_transform(y.reshape(-1, 1))

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_onehot, test_size=0.2, random_state=42)

# 3. Arquitectura del Modelo
model = Sequential([
    Dense(16, activation='relu', input_shape=(4,)),
    Dense(8, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Entrenar
history = model.fit(X_train, y_train, epochs=80, batch_size=8, validation_split=0.1, verbose=0)
print("Red Neuronal entrenada exitosamente.")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Red Neuronal entrenada exitosamente.


# 2. Funciones para Generar Gráficas y Métricas

Crearemos funciones auxiliares que generen las gráficas necesarias para nuestro Dashboard de Gradio.

In [ ]:
def plot_univariate():
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()
    for i, col in enumerate(feature_names):
        sns.histplot(data=df, x=col, hue='species', kde=True, ax=axes[i], element="step")
        axes[i].set_title(f'Distribución de {col}')
    plt.tight_layout()
    return fig

def plot_bivariate():
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=df, x='sepal length (cm)', y='petal length (cm)', hue='species', style='species', s=100, ax=ax)
    ax.set_title('Relación: Sepal Length vs Petal Length')
    return fig

def plot_multivariate():
    # Usamos pairplot y guardamos su figura
    g = sns.pairplot(df, hue='species', palette='Set2')
    g.fig.suptitle('Análisis Multivariado - Pairplot', y=1.02)
    return g.fig

def plot_correlation():
    fig, ax = plt.subplots(figsize=(7, 5))
    numeric_df = df.drop(columns=['species'])
    sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, ax=ax)
    ax.set_title('Matriz de Correlación de Características')
    return fig

def plot_metrics():
    # Predicciones
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)

    # 1. Matriz de confusión
    cm = confusion_matrix(y_true_classes, y_pred_classes)

    # 2. Histograma de Errores
    errors = y_pred - y_test

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Graficar Matriz de Confusión
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names, ax=axes[0])
    axes[0].set_title('Matriz de Confusión')
    axes[0].set_xlabel('Predicho')
    axes[0].set_ylabel('Verdadero')

    # Graficar Histograma de Errores (Diferencia de probabilidades esperadas vs reales)
    sns.histplot(errors.flatten(), bins=15, kde=True, ax=axes[1], color='crimson')
    axes[1].set_title('Distribución del Histograma de Errores (Residuales)')
    axes[1].set_xlabel('Error')

    plt.tight_layout()

    # Generar reporte de clasificación en texto plano
    report_str = classification_report(y_true_classes, y_pred_classes, target_names=target_names)

    return fig, report_str

# 3. Implementación de la Interfaz Gradio con Pestañas

Ahora creamos una interfaz unificada utilizando `gr.Blocks`. Configuraremos diferentes pestañas para visualizar el Dataframe, las estadísticas descriptivas, los diferentes gráficos del EDA, la evaluación de la Red Neuronal, un clasificador interactivo para nuevas muestras y las conclusiones finales.

In [ ]:
import gradio as gr

# Generar de antemano el reporte de métricas del modelo
fig_metrics, report_text = plot_metrics()
plt.close(fig_metrics) # Prevenir renderizado doble en colab

def predict_species(sepal_l, sepal_w, petal_l, petal_w):
    # Escalar entrada
    raw_features = np.array([[sepal_l, sepal_w, petal_l, petal_w]])
    scaled_features = scaler.transform(raw_features)
    # Predecir
    prediction = model.predict(scaled_features)
    pred_class = np.argmax(prediction, axis=1)[0]
    confidence = float(prediction[0][pred_class])

    # Crear un diccionario de probabilidades de retorno para gradio
    return {target_names[i]: float(prediction[0][i]) for i in range(3)}

# Definir el tema y diseño de Gradio
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌸 Dashboard Analítico de Iris & Clasificación por Red Neuronal")

    with gr.Tabs():
        # Pestaña 1: Resumen del Ejercicio
        with gr.Tab("Descripción del Ejercicio"):
            gr.Markdown("""
            ### Contexto del Ejercicio
            Este dashboard integra un **Análisis Exploratorio de Datos (EDA)** interactivo con un modelo predictivo basado en una **Red Neuronal Artificial (ANN)** utilizando el dataset Iris.

            **El dataset contiene 3 especies de flores de Iris:**
            - *Iris-setosa*
            - *Iris-versicolor*
            - *Iris-virginica*

            **Características (Atributos):**
            1. Sepal Length (Largo del Sépalo en cm)
            2. Sepal Width (Ancho del Sépalo en cm)
            3. Petal Length (Largo del Pétalo en cm)
            4. Petal Width (Ancho del Pétalo en cm)

            Navega por las diferentes pestañas para interactuar con los datos, explorar gráficos univariados, bivariados y multivariados, evaluar el modelo predictivo e inferir sobre nuevas muestras en tiempo real.
            """)

        # Pestaña 2: Visualización de Datos (EDA)
        with gr.Tab("Datos & Estadísticas"):
            gr.Markdown("### Tabla de Muestras del Dataset")
            gr.DataFrame(df.head(20))
            gr.Markdown("### Estadísticas Descriptivas")
            gr.DataFrame(df.describe())

        # Pestaña 3: Distribuciones Univariadas
        with gr.Tab("Visualizaciones Univariadas"):
            gr.Markdown("### Distribuciones Individuales de las Características")
            gr.Plot(plot_univariate())

        # Pestaña 4: Visualizaciones Bivariadas & Multivariadas
        with gr.Tab("Relaciones Bi y Multivariadas"):
            gr.Markdown("### Gráfico de Dispersión Bivariado")
            gr.Plot(plot_bivariate())
            gr.Markdown("### Gráfico Multivariado (Pairplot Completo)")
            gr.Plot(plot_multivariate())

        # Pestaña 5: Matriz de Correlación
        with gr.Tab("Correlaciones"):
            gr.Markdown("### Matriz de Correlación de Pearson")
            gr.Plot(plot_correlation())

        # Pestaña 6: Rendimiento de la Red Neuronal
        with gr.Tab("Métricas de la Red Neuronal"):
            gr.Markdown("### Evaluación de Rendimiento del Modelo")
            with gr.Row():
                with gr.Column():
                    gr.Markdown("**Reporte de Clasificación:**")
                    gr.Text(report_text, label="Métricas")
                with gr.Column():
                    gr.Markdown("**Matriz de Confusión e Histograma de Errores:**")
                    gr.Plot(fig_metrics)

        # Pestaña 7: Prueba de Clasificación (Demo Individual)
        with gr.Tab("Predicción Individual (Demo)"):
            gr.Markdown("### Clasificación en Tiempo Real de una nueva flor")
            with gr.Row():
                with gr.Column():
                    sl = gr.Slider(minimum=4.0, maximum=8.0, value=5.8, label="Sepal Length (cm)")
                    sw = gr.Slider(minimum=2.0, maximum=4.5, value=3.0, label="Sepal Width (cm)")
                    pl = gr.Slider(minimum=1.0, maximum=7.0, value=4.35, label="Petal Length (cm)")
                    pw = gr.Slider(minimum=0.1, maximum=2.5, value=1.3, label="Petal Width (cm)")
                    btn_predict = gr.Button("Clasificar Especie")
                with gr.Column():
                    label_out = gr.Label(num_top_classes=3, label="Predicción de la Red Neuronal")

            btn_predict.click(
                fn=predict_species,
                inputs=[sl, sw, pl, pw],
                outputs=label_out
            )

        # Pestaña 8: Conclusiones
        with gr.Tab("Conclusiones"):
            gr.Markdown("""
            ### Conclusiones del EDA e Insights:
            1. **Separabilidad de Especies**: La especie *Iris-setosa* se separa de forma prácticamente lineal de las demás especies, especialmente en el largo y ancho del pétalo. Las especies *versicolor* y *virginica* presentan un solapamiento parcial, lo que requiere de límites de decisión no lineales.
            2. **Fuerte Correlación**: Existe una correlación positiva sumamente alta entre el largo y ancho de los pétalos (0.96), lo que indica que estas dos métricas describen dinámicas muy similares en la estructura del dataset.

            ### Conclusiones del Entrenamiento de la Red Neuronal:
            1. **Alta Precisión**: El modelo de Red Neuronal Artificial converge rápidamente en pocas épocas dada la simplicidad del dataset, logrando una precisión superior al 95% en el set de prueba.
            2. **Análisis de Errores**: La matriz de confusión muestra que los escasos fallos de clasificación ocurren casi exclusivamente en la frontera difusa entre *Iris-versicolor* e *Iris-virginica*.
            3. **Eficacia**: Gradio permite democratizar modelos de Machine Learning y Deep Learning complejos en interfaces sencillas y de fácil interpretación para el usuario final.
            """)

# Lanzar la aplicación
demo.launch(share=True)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step


/tmp/ipykernel_3657/1281221923.py:20: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7c80076c5ad128e75.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 🛠️ Instalar Dependencias para Streamlit y Ngrok

Instalamos `streamlit` para construir la nueva interfaz interactiva y `pyngrok` para gestionar el túnel de conexión segura.

In [ ]:
%%capture
!pip install streamlit pyngrok

### 📝 Crear la Aplicación en Streamlit (`streamlit_app.py`)

Escribimos el código de la aplicación. Mantendremos el mismo modelo entrenado previamente y adaptaremos la interfaz con pestañas interactivas, sliders y gráficos usando la API nativa de Streamlit.

In [ ]:
%%writefile streamlit_app.py
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

st.set_page_config(page_title="Iris & Red Neuronal", layout="wide")

# 1. Carga de datos y entrenamiento
sns.set_theme(style="whitegrid")

@st.cache_resource
def load_data_and_train():
    iris = load_iris()
    X = iris.data
    y = iris.target
    feature_names = iris.feature_names
    target_names = iris.target_names

    df = pd.DataFrame(X, columns=feature_names)
    df['species'] = [target_names[i] for i in y]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    encoder = OneHotEncoder(sparse_output=False)
    y_onehot = encoder.fit_transform(y.reshape(-1, 1))

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_onehot, test_size=0.2, random_state=42)

    model = Sequential([
        Dense(16, activation='relu', input_shape=(4,)),
        Dense(8, activation='relu'),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=80, batch_size=8, validation_split=0.1, verbose=0)

    return df, scaler, model, target_names, feature_names, X_test, y_test

df, scaler, model, target_names, feature_names, X_test, y_test = load_data_and_train()

st.title("🌸 Dashboard Analítico de Iris & Clasificación por Red Neuronal")

tabs = st.tabs([
    "Descripción",
    "Datos & Estadísticas",
    "Visualizaciones",
    "Métricas del Modelo",
    "Predicción (Demo)"
])

with tabs[0]:
    st.subheader("Contexto del Ejercicio")
    st.markdown("""
    Este dashboard integra un **Análisis Exploratorio de Datos (EDA)** interactivo con un modelo predictivo basado en una **Red Neuronal Artificial (ANN)** utilizando el dataset Iris.
    """)

with tabs[1]:
    st.subheader("Tabla de Muestras")
    st.dataframe(df.head(20))
    st.subheader("Estadísticas Descriptivas")
    st.dataframe(df.describe())

with tabs[2]:
    st.subheader("Análisis Gráfico")
    opcion_grafico = st.selectbox("Selecciona un gráfico:", [
        "Distribuciones Individuales",
        "Relación Sépalo vs Pétalo",
        "Análisis Multivariado",
        "Matriz de Correlación"
    ])

    fig, ax = plt.subplots(figsize=(8, 5))
    if opcion_grafico == "Distribuciones Individuales":
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        axes = axes.flatten()
        for i, col in enumerate(feature_names):
            sns.histplot(data=df, x=col, hue='species', kde=True, ax=axes[i], element="step")
        st.pyplot(fig)
    elif opcion_grafico == "Relación Sépalo vs Pétalo":
        sns.scatterplot(data=df, x='sepal length (cm)', y='petal length (cm)', hue='species', s=100, ax=ax)
        st.pyplot(fig)
    elif opcion_grafico == "Análisis Multivariado":
        g = sns.pairplot(df, hue='species', palette='Set2')
        st.pyplot(g.fig)
    elif opcion_grafico == "Matriz de Correlación":
        numeric_df = df.drop(columns=['species'])
        sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', ax=ax)
        st.pyplot(fig)

with tabs[3]:
    st.subheader("Rendimiento de la Red Neuronal")
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)

    col1, col2 = st.columns(2)
    with col1:
        st.text("Reporte de Clasificación:")
        st.text(classification_report(y_true_classes, y_pred_classes, target_names=target_names))
    with col2:
        fig, ax = plt.subplots(figsize=(5, 4))
        cm = confusion_matrix(y_true_classes, y_pred_classes)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names, ax=ax)
        st.pyplot(fig)

with tabs[4]:
    st.subheader("Clasificación en Tiempo Real")
    col1, col2 = st.columns(2)
    with col1:
        sl = st.slider("Sepal Length (cm)", 4.0, 8.0, 5.8)
        sw = st.slider("Sepal Width (cm)", 2.0, 4.5, 3.0)
        pl = st.slider("Petal Length (cm)", 1.0, 7.0, 4.35)
        pw = st.slider("Petal Width (cm)", 0.1, 2.5, 1.3)
    with col2:
        raw_features = np.array([[sl, sw, pl, pw]])
        scaled_features = scaler.transform(raw_features)
        prediction = model.predict(scaled_features)[0]

        res_dict = {target_names[i]: float(prediction[i]) for i in range(3)}
        st.write("Probabilidades:")
        st.json(res_dict)
        st.success(f"Especie Predicha: **{target_names[np.argmax(prediction)]}**")

Overwriting streamlit_app.py


### 🚀 Configurar Ngrok y Lanzar Streamlit

Ingresa tu token de Ngrok en la variable `NGROK_AUTH_TOKEN` de abajo y ejecuta la celda para obtener tu enlace de acceso público y gratuito.

In [ ]:
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError
import subprocess
import time

# CONFIGURACIÓN
NGROK_AUTH_TOKEN = "3LcEOsbFSXcZmYcznp4aDC0aDCh_2s3SfyTd99iGcKggkHnfM" # <--- Tu Authtoken de ngrok

# 1. Autenticar con el Token
if NGROK_AUTH_TOKEN != "TU_NGROK_TOKEN_AQUÍ":
    ngrok.kill()
    try:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    except PyngrokNgrokError as e:
        # Si el token ya estaba guardado, ignoramos el error y continuamos con el túnel
        if "Authtoken already set" in str(e):
            print("ℹ️ El token de Ngrok ya estaba configurado en el sistema. Continuando...")
        else:
            raise e

    # 2. Iniciar el servidor de Streamlit en segundo plano
    print("Iniciando el servidor de Streamlit...")
    process = subprocess.Popen(["streamlit", "run", "streamlit_app.py", "--server.port", "8501"])
    time.sleep(4)

    # 3. Crear el túnel con Ngrok apuntando al puerto de Streamlit
    print("Estableciendo el túnel de Ngrok...")
    public_url = ngrok.connect(8501, "http")
    print(f"\n🎉 ¡Servidor Streamlit listo de forma pública!\n🔗 Enlace de acceso: {public_url}\n")
else:
    print("⚠️ Por favor, ingresa tu Authtoken de Ngrok en la variable NGROK_AUTH_TOKEN.")

ℹ️ El token de Ngrok ya estaba configurado en el sistema. Continuando...
Iniciando el servidor de Streamlit...
Estableciendo el túnel de Ngrok...

🎉 ¡Servidor Streamlit listo de forma pública!
🔗 Enlace de acceso: NgrokTunnel: "https://bagel-busboy-undertake.ngrok-free.dev" -> "http://localhost:8501"

